# LensForge Baseline

This notebook is a reproducible starting point for the DeepLense `Lens Finding & Data Pipelines` task.

Goals:
- inspect the class-imbalanced dataset
- train a threshold-tuned PyTorch baseline
- review validation and test ROC-AUC metrics


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("tmp/matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
DATA_ROOT = Path("data/lens-finding-test")
REPORT_PATH = Path("reports/notebook_metrics.json")
MODEL_PATH = Path("models/notebook_best_model.pt")

summary_rows = []
for split in ["train_lenses", "train_nonlenses", "test_lenses", "test_nonlenses"]:
    files = sorted((DATA_ROOT / split).glob("*.npy"))
    sample = np.load(files[0])
    summary_rows.append({
        "split": split,
        "count": len(files),
        "shape": tuple(sample.shape),
        "dtype": str(sample.dtype),
        "min": float(sample.min()),
        "max": float(sample.max()),
    })

pd.DataFrame(summary_rows)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
examples = [
    ("train_lenses", sorted((DATA_ROOT / "train_lenses").glob("*.npy"))[0]),
    ("train_nonlenses", sorted((DATA_ROOT / "train_nonlenses").glob("*.npy"))[0]),
]

for row, (label, path) in enumerate(examples):
    image = np.load(path)
    for channel in range(3):
        axes[row, channel].imshow(image[channel], cmap="magma")
        axes[row, channel].set_title(f"{label} | channel {channel}")
        axes[row, channel].axis("off")

plt.tight_layout()
plt.show()


## Baseline run

The command below keeps the run intentionally lightweight for notebook use while using LensForge's current best imbalance strategy. Increase `--train-fraction` to `1.0`, `--test-fraction` to `1.0`, and the number of epochs for a full training run.


In [ ]:
command = [
    "python3",
    "train.py",
    "--data-root",
    str(DATA_ROOT),
    "--epochs",
    "3",
    "--batch-size",
    "128",
    "--train-fraction",
    "0.25",
    "--balance-strategy",
    "both",
    "--test-fraction",
    "0.25",
    "--report-path",
    str(REPORT_PATH),
    "--model-path",
    str(MODEL_PATH),
]

completed = subprocess.run(command, check=True, text=True, capture_output=True)
print(completed.stdout)


In [ ]:
report = json.loads(REPORT_PATH.read_text())
pd.DataFrame([
    {"split": "best_validation", **report["best_validation"]},
    {"split": "test", **report["test"]},
])


In [ ]:
history = pd.DataFrame([
    {
        "epoch": row["epoch"],
        "train_loss": row["train"]["loss"],
        "train_auc": row["train"]["roc_auc"],
        "val_loss": row["validation"]["loss"],
        "val_auc": row["validation"]["roc_auc"],
        "val_pr_auc": row["validation"]["pr_auc"],
    }
    for row in report["history"]
])
history


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train")
axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_auc"], marker="o", label="train ROC-AUC")
axes[1].plot(history["epoch"], history["val_auc"], marker="o", label="validation ROC-AUC")
axes[1].plot(history["epoch"], history["val_pr_auc"], marker="o", label="validation PR-AUC")
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title("AUC Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame([
    {"split": "best_validation", "tp": report["best_validation"]["tp"], "fp": report["best_validation"]["fp"], "tn": report["best_validation"]["tn"], "fn": report["best_validation"]["fn"], "threshold": report["best_validation"]["threshold"]},
    {"split": "test", "tp": report["test"]["tp"], "fp": report["test"]["fp"], "tn": report["test"]["tn"], "fn": report["test"]["fn"], "threshold": report["test"]["threshold"]},
])


In [ ]:
val_curves = report["best_validation_curves"]
test_curves = report["test_curves"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(val_curves["roc_curve"]["fpr"], val_curves["roc_curve"]["tpr"], label="validation")
axes[0].plot(test_curves["roc_curve"]["fpr"], test_curves["roc_curve"]["tpr"], label="test")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].plot(val_curves["pr_curve"]["recall"], val_curves["pr_curve"]["precision"], label="validation")
axes[1].plot(test_curves["pr_curve"]["recall"], test_curves["pr_curve"]["precision"], label="test")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
error_summary = json.loads(Path("reports/error_summary.json").read_text())
pd.DataFrame([error_summary["counts"]]).T.rename(columns={0: "count"})


## Error analysis

The current model recovers most positives at the tuned threshold, but its main weakness is a relatively large set of high-confidence false positives on non-lens images.


In [ ]:
top_fp = pd.DataFrame(error_summary["top_false_positives"])[["path", "probability", "outcome"]]
top_fn = pd.DataFrame(error_summary["top_false_negatives"])[["path", "probability", "outcome"]]
display(top_fp)
display(top_fn)


In [ ]:
def show_examples(rows, title):
    rows = list(rows)
    if not rows:
        print(f"No examples available for {title.lower()}.")
        return
    fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
    if len(rows) == 1:
        axes = np.expand_dims(axes, axis=0)
    for row_idx, row in enumerate(rows):
        image = np.load(row["path"])
        for channel in range(3):
            axes[row_idx, channel].imshow(image[channel], cmap="magma")
            axes[row_idx, channel].set_title(f"{title} #{row_idx + 1} | ch {channel} | p={row['probability']:.3f}")
            axes[row_idx, channel].axis("off")
    plt.tight_layout()
    plt.show()

show_examples(error_summary["top_false_positives"][:3], "False positive")
show_examples(error_summary["top_false_negatives"][:3], "False negative")


## BCE vs focal loss

A focused comparison shows that focal loss reduces false positives, but the current configuration loses too much recall and overall ROC/PR performance compared with BCE.


In [ ]:
focal_summary = pd.DataFrame(json.loads(Path("reports/focal_loss_summary.json").read_text()))
focal_summary


## Current best setting

On the current comparison slice, `balance-strategy=both` performed best, combining sampler balancing, loss weighting, and validation threshold tuning.


## Next steps

- run a longer training schedule on the full training split
- compare sampler-based balancing with focal loss
- add notebook commentary on architectural choices and error analysis
